# 生成式视觉大模型：从 Diffusion 到文生图与文生视频

> **本章定位**：从条件潜空间扩散的可验证基线，扩展到 Diffusion Transformer（DiT）、Rectified Flow、文生图、图像编辑、文生视频与世界模型。

> **章节边界**：本章属于跨方向专题：视觉生成，以 `30` 的 Attention 机制为基础，聚焦生成式视觉模型；图像分类与感知型多模态模型分别由 `E40_cv_vit.ipynb` 与 `E20_multimodal_llm.ipynb` 承接。

**本章总览**：Stable Diffusion 的 VAE、文本编码器、条件 U-Net、Scheduler 与 Classifier-Free Guidance 构成扩散数据流基线；随后扩展到 DiT、Flow Matching、视频时空 Latent、开放权重模型图谱和生产验收。

<!-- diagram:stable-diffusion-overview -->
Stable Diffusion 在潜空间中迭代去噪，并用文本表示通过 Cross-Attention 提供条件：

![架构图：Stable Diffusion 文本条件、潜空间去噪循环、Scheduler 与 VAE 解码架构](assets/figures/E50_cv_diffusion/stable-diffusion-overview.svg)

[TikZ 源文件](assets/figures/E50_cv_diffusion/stable-diffusion-overview.tex)


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 跨方向专题：视觉生成 |
| 本章定位 | 从条件潜空间扩散建立共同原理，并比较文生图、编辑、文生视频、音视频与世界模型。 |
| 先修知识 | 建议完成 `30`；理解概率噪声、卷积和 Cross-Attention。 |
| 预计时间 | 3～4 小时 |
| 运行资源 | 核心张量实验可在 CPU 运行；真实 Pipeline 建议使用 GPU。 |
| 输入 | 官方模型卡与许可证，以及图像或视频 Latent、噪声或流时间、文本与参考媒体条件。 |
| 交付物 | 生成模型图谱、最小扩散核心、Scheduler/U-Net/VAE/DiT 对照和任务级 Pipeline 验收矩阵。 |

### 1.1．学习目标

完成本章后，读者能够解释加噪、时间条件、噪声预测、CFG、VAE 与反向采样的连续数据流，并能将图像扩散基线迁移到 DiT、Flow Matching 和视频生成的时空 Latent 契约。


### 1.2．环境与依赖

本章在 Python 3.12.13、PyTorch 2.11.0、Diffusers 0.39.0、Transformers 5.13.1 与 Accelerate 1.14.0 环境下验证。Stable Diffusion v1.5 的 `512×512`、FP16、Batch 1 推理通常需要 8 GiB 级显存起步，但峰值仍受 Attention 实现、VAE、Scheduler 与运行时工作区影响；新模型应依据各自兼容矩阵实测。


## 2．直觉与输入输出契约

模型选择以输出制品与任务契约为起点：

| 任务 | 输入 → 输出 | 主要验收维度 |
|---|---|---|
| 文生图 | 文本 → 图像 | 提示遵循、文字/布局、画质、多样性、安全、延迟 |
| 图像编辑 | 文本 + 参考图/Mask → 图像 | 指令遵循、未编辑区域保持、身份/风格一致、可逆与溯源 |
| 文/图生视频 | 文本和/或首帧 → 视频 | 运动、身份、镜头、时序、物理、清晰度、帧率/时长 |
| 音视频联合 | 文本/媒体 → 视频 + 音频 | 上述维度 + 声画同步、语音/音乐版权与声音冒充 |
| 世界模型 | 文本/视觉/动作 → 未来状态/视频/动作 | 动力学、可控性、闭环稳定、覆盖率与安全分析 |

<!-- diagram:generative-vision-contract -->

![架构图：图像与视频生成中多条件编码、去噪或流求解、Latent 与 VAE 输出契约](assets/figures/E50_cv_diffusion/generative-vision-contract.svg)

[TikZ 源文件](assets/figures/E50_cv_diffusion/generative-vision-contract.tex)

### 2.1．三条生成架构路线

| 路线 | 稳定数据流 | 典型优点 | 主要代价 |
|---|---|---|---|
| U-Net Latent Diffusion | 多尺度卷积/注意力预测噪声或速度 | 层级局部归纳偏置强、生态成熟 | 结构复杂，超大规模和多模态联合不如 Transformer 统一 |
| Diffusion Transformer / Flow | 把 Latent Patch 化，以 Transformer 预测速度/流 | 易扩展、易与文本/图像 Token 联合注意 | 时空 Token 带来显存与计算压力，依赖高效 Attention/并行 |
| 自回归视觉 Token | 离散图像/视频 Token 逐步生成 | 与 LLM 统一、理解与生成可共享主干 | 序列长、采样慢，离散 Tokenizer 上限影响画质 |

本章采用 U-Net Latent Diffusion 的原理实现，以逐项验证加噪、训练目标、条件、采样和 VAE 契约；所得证据进一步迁移到 DiT 与 Flow，而不将 Stable Diffusion v1 视为唯一生产选择。


### 2.2．开放权重生成模型图谱（2026-08-09 快照）

#### 2.2.1．文生图与编辑

| 地区/家族 | 任务与架构视角 | 开放状态 | 生产边界 |
|---|---|---|---|
| 中国｜**Qwen-Image 1.x / Qwen-Image-2.0** | 1.x 以 MMDiT 文生图和独立 Edit 分支为主；2.0 在单一模式中统一生成与编辑，并继续强化文字/布局 | Apache-2.0 开放权重 | 必须按代际选择 Pipeline；Layered 等能力仍按独立制品核验。重模型需显存规划，输出仍需审核与来源追踪 |
| 中国｜**Janus-Pro** | 图像理解 + 自回归视觉 Token 生成，共享 Transformer、解耦视觉编码器 | 代码 MIT，权重为 DeepSeek Model License | 适合统一建模研究；分辨率和画质目标与专用生产生图模型不同 |
| 全球｜**FLUX** | Rectified-flow Transformer；schnell 面向快速生成，dev/新代覆盖高质量与编辑 | schnell 为 Apache-2.0；dev 检查点常为非商业/自定义许可 | 必须区分“输出可用”和“权重可商用”；仓库代码许可不能替代模型许可 |
| 全球｜**Stable Diffusion 3.5** | MMDiT + 多文本编码器 | 代码 MIT，权重为 Stability AI Community License | 商业门槛、AUP、归因和衍生训练限制按当前许可核验 |

#### 2.2.2．文生视频、音视频与世界模型

| 地区/家族 | 任务与架构视角 | 开放状态 | 生产边界 |
|---|---|---|---|
| 中国｜**Wan2.2** | T2V/I2V；视频 DiT，A14B 使用高/低噪声双专家，另有较轻 TI2V 路线 | Apache-2.0 开放权重 | 视频秒数仍对应高时延；不同分支能力、显存与质量不能混写 |
| 中国｜**HunyuanVideo-1.5** | T2V/I2V；DiT、时空 VAE、选择性/滑窗注意力与超分 | Tencent Hunyuan Community License | 含地域、规模或用途限制，不能称无条件可商用 |
| 中国｜**CogVideoX** | T2V/I2V；3D Causal VAE + Video Transformer | 代码 Apache-2.0；不同大小权重许可不同 | 2B 与 5B/I2V 的许可和资源边界需分开核验 |
| 美国｜**Mochi 1** | 文生视频；AsymmDiT + AsymmVAE | Apache-2.0 | 官方参考实现显存重；快速运动、形变和风格覆盖需业务评测 |
| 美国｜**NVIDIA Cosmos 3** | 理解 + 图像/视频/声音/动作生成的世界模型；AR Reasoner + Diffusion Generator | OpenMDW-1.1 | Edge/工作站/数据中心规格分层；物理、3D、时序或声画错误禁止直接进入安全关键控制 |
| 全球｜**LTX-2** | 视频 + 同步音频；非对称双流 Diffusion Transformer | 自定义 Community License | 收入门槛、生成标识和竞争性用途限制需法务核验；旧 LTX-Video 许可不能继承到 LTX-2 |

表中的开放状态只是可见性摘要，不是线性许可证等级。生产清单必须另列：权重获取、代码、修改、再分发、商业部署、MaaS、地域、收入/用户门槛、衍生训练、用输出训练、归因与 AUP。Sora、Veo、GPT 图像、Runway 等只能列为闭源/API 对照，不进入开放权重表。


### 2.3．资源条件与模型选择

Stable Diffusion v1.5 仅作为稳定接口基线。图谱中的新模型具有不同的文本编码器、VAE、DiT、分辨率与时空 Token 契约，资源预算应分别读取对应 Config，并在目标硬件上测量峰值显存、延迟和吞吐。CPU 可验证核心张量语义，但不适合作为生成性能基线。


In [ ]:
# 导入本章原理实现与 Diffusers 对照所需依赖。


In [ ]:
# 固定随机状态并选择设备，建立扩散模型实验环境。

import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

# 3407 仅固定噪声、时间步与 Latent 采样；质量评测使用预注册 Seed 集合，跨设备或 Scheduler 不保证像素一致。
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 选择当前可用设备，后续张量和模型统一放到同一计算后端。
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
print("PyTorch:", torch.__version__, "| device:", DEVICE)


<!-- theory-math-contract:v1 -->
### 2.4．核心机制的语言与数学表达

前向扩散用随时间增加的高斯噪声把数据逐步变为近似标准正态分布，并训练网络预测噪声或等价参数化：

$$
q(x_t\mid x_0)=\mathcal N\!\left(\sqrt{\bar\alpha_t}x_0,(1-\bar\alpha_t)I\right),\qquad
x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\,\epsilon
$$

$$
\mathcal L_{\epsilon}=\mathbb E_{x_0,t,\epsilon}\left[\|\epsilon-\epsilon_\theta(x_t,t,c)\|_2^2\right]
$$

其中，$x_0$ 是干净样本，$t$ 是时间步，$\bar\alpha_t$ 是累计信号保留率，$\epsilon\sim\mathcal N(0,I)$，$c$ 是可选条件。`my_add_noise`、噪声预测器与 Diffusers Scheduler 分别对应采样公式、学习目标和生产步进协议。不同 Scheduler 或 $v$-prediction 会改变训练/采样参数化，不能只替换函数名。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．前向扩散：逐步加噪

定义 $\beta_t$ 为每步噪声强度，$\alpha_t=1-\beta_t$，$\bar\alpha_t=\prod_{s=1}^{t}\alpha_s$。借助重参数化，可一步从干净样本得到任意时刻：

$$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,\quad \epsilon\sim\mathcal N(0,I)$$

#### 3.1.1．噪声日程与前向加噪的从零实现

<!-- diagram:diffusion-forward-training-pair -->
训练样本不是逐步加噪生成，而是直接采样时间步并构造对应的带噪状态：

![架构图：扩散训练中直接采样时间步构造带噪样本并监督噪声预测](assets/figures/E50_cv_diffusion/diffusion-forward-training-pair.svg)

[TikZ 源文件](assets/figures/E50_cv_diffusion/diffusion-forward-training-pair.tex)


In [ ]:
# 生成噪声调度，并按闭式公式把干净样本加噪到任意时间步。

def my_linear_beta_schedule(num_steps: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    """生成从 beta_start 到 beta_end 的一维线性扩散噪声日程。"""
    return torch.linspace(beta_start, beta_end, num_steps)

def my_extract(values: torch.Tensor, timesteps: torch.Tensor, shape: torch.Size):
    """按批内时间步提取一维日程系数，并重塑为可与目标张量广播的形状。"""
    # 为批内每个样本选择自己的时间步系数，并扩展到可广播形状。
    out = values.to(timesteps.device).gather(0, timesteps)
    return out.view(timesteps.shape[0], *((1,) * (len(shape) - 1)))

def my_add_noise(x0, noise, timesteps, alpha_bars):
    """使用 DDPM 闭式公式把干净样本与高斯噪声混合到指定时间步，输出形状与 x0 相同。"""
    # alpha_bar 控制保留多少原始信号，以及注入多少高斯噪声。
    sqrt_signal = my_extract(alpha_bars.sqrt(), timesteps, x0.shape)
    sqrt_noise = my_extract((1 - alpha_bars).sqrt(), timesteps, x0.shape)
    return sqrt_signal * x0 + sqrt_noise * noise

# 1000 步及 beta 1e-4→0.02 对齐经典 DDPM 线性日程；训练 timestep、Scheduler 与 prediction_type 必须成套变更。
num_steps = 1000
betas = my_linear_beta_schedule(num_steps)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
# 固定形状：x0.shape = [4, 1, 16, 16]。
x0 = torch.zeros(4, 1, 16, 16)
x0[:, :, 4:12, 4:12] = 1.0
noise = torch.randn_like(x0)
timesteps = torch.tensor([0, 100, 500, 999])
xt = my_add_noise(x0, noise, timesteps, alpha_bars)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for axis, image, timestep in zip(axes, xt, timesteps):
    axis.imshow(image.squeeze().numpy(), cmap="gray")
    axis.set_title(f"t={int(timestep)}")
    axis.axis("off")
plt.tight_layout()


#### 3.1.2．`DDPMScheduler.add_noise` 等价性验证

Scheduler 集中管理噪声日程、预测类型、方差和采样步进。训练与推理必须使用兼容配置。


In [ ]:
# 使用 DDPMScheduler 执行相同的前向加噪过程。

from diffusers import DDPMScheduler

# 配置调度器，控制训练或采样过程中随步骤变化的参数。
lib_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_start=1e-4,
    beta_end=0.02,
    beta_schedule="linear",
    prediction_type="epsilon",
)
lib_xt = lib_scheduler.add_noise(x0, noise, timesteps)
# FP32 门禁使用 rtol=1e-5、atol=1e-6；更换低精度、设备或日程后按参考误差分布重设。
torch.testing.assert_close(lib_xt, xt, rtol=1e-5, atol=1e-6)
print("max error:", float((lib_xt - xt).abs().max()))


#### 3.1.3．噪声旋钮：信号与噪声如何交换主导权

学习问题是：时间步增大时，$x_t$ 为什么逐渐由噪声主导。为隔离时间步这一项，下面固定上一单元 Batch 中第 0 个 `x0` 与第 0 个 `noise`，把同一份样本和同一份 $\epsilon$ 复制到四个 `timesteps`，再通过正文的 `my_add_noise` 与 `DDPMScheduler.add_noise` 重算同一条 Trace。每一列保持样本身份不变：第一行是 $\sqrt{\bar\alpha_t}x_0$，第二行是 $\sqrt{1-\bar\alpha_t}\epsilon$，第三行是两者逐元素相加后的 $x_t$。

验收条件是两个系数的平方和为 1，两个分量之和精确重构正文的 `xt`，并继续与 `DDPMScheduler.add_noise` 对齐。颜色范围在全部分量之间统一，避免自动缩放把微弱信号显示得与主导分量同样强。


In [ ]:
# 使用正文真实张量分解前向扩散公式，并以固定色标呈现信噪交换。
visual_timesteps = timesteps.detach().cpu()
visual_trace_size = int(visual_timesteps.numel())
visual_x0 = x0[:1].detach().float().cpu().repeat(visual_trace_size, 1, 1, 1)
visual_noise = noise[:1].detach().float().cpu().repeat(visual_trace_size, 1, 1, 1)
trace_x0_identity_error = float((visual_x0 - visual_x0[:1]).abs().max())
trace_noise_identity_error = float((visual_noise - visual_noise[:1]).abs().max())
if trace_x0_identity_error != 0.0 or trace_noise_identity_error != 0.0:
    raise RuntimeError("噪声旋钮没有保持同一份 x0 与 epsilon")
visual_alpha_bar = alpha_bars[visual_timesteps].detach().float().cpu()
signal_coeff = visual_alpha_bar.sqrt().view(-1, 1, 1, 1)
noise_coeff = (1.0 - visual_alpha_bar).sqrt().view(-1, 1, 1, 1)
signal_component = signal_coeff * visual_x0
noise_component = noise_coeff * visual_noise
reconstructed_xt = signal_component + noise_component
visual_xt = my_add_noise(
    visual_x0, visual_noise, visual_timesteps, alpha_bars
).detach().float().cpu()
visual_lib_xt = lib_scheduler.add_noise(
    visual_x0, visual_noise, visual_timesteps
).detach().float().cpu()

coefficient_error = float(
    ((signal_coeff.square() + noise_coeff.square()) - 1.0).abs().max()
)
reconstruction_error = float((reconstructed_xt - visual_xt).abs().max())
scheduler_error = float((visual_lib_xt - visual_xt).abs().max())
if coefficient_error > 1e-6:
    raise RuntimeError(f"信号与噪声系数不满足单位圆契约：{coefficient_error:.2e}")
if reconstruction_error > 1e-6:
    raise RuntimeError(f"两个分量未能重构 x_t：{reconstruction_error:.2e}")
if scheduler_error > 1e-5:
    raise RuntimeError(f"原理实现与 Scheduler 超出 FP32 门禁：{scheduler_error:.2e}")

snr = visual_alpha_bar / (1.0 - visual_alpha_bar).clamp_min(1e-12)
all_components = torch.cat(
    [signal_component, noise_component, reconstructed_xt], dim=0
)
color_limit = float(all_components.abs().max())
DIFFUSION_VIZ = {
    "background": "#0B1020",
    "foreground": "#E6EDF7",
    "muted": "#8792A8",
    "signal": "#4EA5D9",
    "noise": "#F2A65A",
    "sum": "#55C1A7",
}

fig, axes = plt.subplots(
    3, len(visual_timesteps), figsize=(12.5, 8.2),
    facecolor=DIFFUSION_VIZ["background"], squeeze=False,
)
rows = [
    (signal_component, DIFFUSION_VIZ["signal"], r"signal: $\sqrt{\bar\alpha_t}x_0$"),
    (noise_component, DIFFUSION_VIZ["noise"], r"noise: $\sqrt{1-\bar\alpha_t}\epsilon$"),
    (reconstructed_xt, DIFFUSION_VIZ["sum"], r"sum: $x_t$"),
]
for column, timestep in enumerate(visual_timesteps.tolist()):
    for row, (images, frame_color, row_label) in enumerate(rows):
        axis = axes[row, column]
        axis.set_facecolor(DIFFUSION_VIZ["background"])
        last_image = axis.imshow(
            images[column, 0], cmap="RdBu_r",
            vmin=-color_limit, vmax=color_limit, interpolation="nearest",
        )
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_color(frame_color)
            spine.set_linewidth(2.0 if row == 2 else 1.2)
        if column == 0:
            axis.set_ylabel(row_label, color=frame_color, fontsize=10)
        if row == 0:
            axis.set_title(
                f"t={timestep}\na={float(signal_coeff[column, 0, 0, 0]):.3f} · "
                f"b={float(noise_coeff[column, 0, 0, 0]):.3f} · SNR={float(snr[column]):.2e}",
                color=DIFFUSION_VIZ["foreground"], fontsize=9, pad=8,
            )

fig.suptitle(
    "Forward diffusion: signal and noise exchange dominance",
    color=DIFFUSION_VIZ["foreground"], fontsize=15, fontweight="bold",
)
fig.text(
    0.5, 0.02,
    "a² + b² = 1 · all panels share one color scale · same x0 and epsilon trace",
    ha="center", color=DIFFUSION_VIZ["muted"], fontsize=9,
)
colorbar = fig.colorbar(
    last_image, ax=axes.ravel().tolist(), shrink=0.72, pad=0.02
)
colorbar.set_label("component value (shared scale)", color=DIFFUSION_VIZ["foreground"])
colorbar.ax.tick_params(colors=DIFFUSION_VIZ["foreground"])
fig.subplots_adjust(left=0.08, right=0.88, bottom=0.08, top=0.86, wspace=0.12, hspace=0.18)
plt.show()

print({
    "source_batch_index": 0,
    "timesteps": visual_timesteps.tolist(),
    "signal_coefficients": [round(float(value), 6) for value in signal_coeff[:, 0, 0, 0]],
    "noise_coefficients": [round(float(value), 6) for value in noise_coeff[:, 0, 0, 0]],
    "snr": [float(value) for value in snr],
    "max_coefficient_error": coefficient_error,
    "same_x0_max_error": trace_x0_identity_error,
    "same_epsilon_max_error": trace_noise_identity_error,
    "max_reconstruction_error": reconstruction_error,
    "max_scheduler_error": scheduler_error,
})


随着时间步增大，信号系数下降、噪声系数上升，SNR 随当前日程降低；第三行必须始终等于前两行的逐元素和。该分镜只解释已知 $x_0$ 与 $\epsilon$ 的前向加噪，不能证明 U-Net 已学会反向去噪，也不能据此判断最终生成质量。


### 3.2．时间步的正弦嵌入与条件注入

U-Net 必须知道当前噪声级别。时间步先映射为多频率正弦/余弦向量，再经过 MLP 注入残差块。

#### 3.2.1．时间嵌入的从零实现


In [ ]:
# 把离散时间步映射为正弦向量，为去噪网络提供噪声阶段信息。

# 32 维用于控制实验成本，10000 沿用正弦位置编码频率底数；维度变化须同步 U-Net 时间投影。
def my_timestep_embedding(timesteps: torch.Tensor, dim: int, max_period: int = 10_000):
    """把 [B] 离散时间步映射为 [B,dim] 正弦嵌入，奇数维时在末尾补零。"""
    half = dim // 2
    # 频率按对数尺度从高到低排列，使不同维度覆盖不同时间范围。
    frequencies = torch.exp(
        -math.log(max_period) * torch.arange(half, device=timesteps.device) / half
    )
    angles = timesteps.float()[:, None] * frequencies[None]
    embedding = torch.cat([torch.cos(angles), torch.sin(angles)], dim=-1)
    if dim % 2:
        embedding = F.pad(embedding, (0, 1))
    return embedding

time_embedding = my_timestep_embedding(torch.tensor([0, 10, 999]), 32)
print(time_embedding.shape)


#### 3.2.2．Diffusers 时间投影接口对照


In [ ]:
# 切换到 Diffusers 的时间投影与 MLP 嵌入模块。

from diffusers.models.embeddings import Timesteps, TimestepEmbedding

# 时间向量由 32 维扩到 128 维（4 倍），用于匹配本章 U-Net 条件通道；修改时须同步所有残差块。
lib_time_projection = Timesteps(num_channels=32, flip_sin_to_cos=True, downscale_freq_shift=0)
lib_time_mlp = TimestepEmbedding(in_channels=32, time_embed_dim=128)
lib_time_embedding = lib_time_mlp(lib_time_projection(torch.tensor([0, 10, 999])))
print(lib_time_embedding.shape)


### 3.3．噪声预测器：带时间条件的 U-Net

U-Net 下采样以获取全局上下文，上采样以恢复空间细节，并通过 skip connection 复用高分辨率特征。Stable Diffusion 还在多个分辨率插入 cross-attention 接收文本条件。

#### 3.3.1．最小可训练 U-Net 的从零实现

<!-- diagram:conditional-unet -->
条件 U-Net 在下采样与上采样之间使用 Skip Connection，并同时接收时间与文本条件：

![架构图：条件 U-Net 的下采样、瓶颈 Cross-Attention、上采样与 Skip Connection](assets/figures/E50_cv_diffusion/conditional-unet.svg)

[TikZ 源文件](assets/figures/E50_cv_diffusion/conditional-unet.tex)


In [ ]:
# 组合残差块、下采样和上采样，构造最小可训练 U-Net。

# GroupNorm 采用 8 组以适配小 Batch，通道数必须整除组数；3×3/padding=1 保持空间尺寸。
# 4×4/stride=2 完成二倍下采样或上采样，修改卷积参数后应复核跳连形状。
class MyResidualBlock(nn.Module):
    """实现由时间条件调制的二维卷积残差块，并在通道变化时投影跳连。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, in_channels: int, out_channels: int, time_dim: int):
        """创建 GroupNorm、卷积、时间投影和可选的通道匹配跳连。"""
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.time = nn.Linear(time_dim, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, x: torch.Tensor, time: torch.Tensor):
        """融合 [B,D] 时间向量与 [B,C,H,W] 特征，返回指定输出通道的同空间尺寸残差特征。"""
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time(F.silu(time))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class MyTinyUNet(nn.Module):
    """实现带单级下采样、瓶颈、上采样和跳连的最小潜空间噪声预测 U-Net。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, channels: int = 32, time_dim: int = 128):
        """创建时间 MLP、编码路径、瓶颈、解码路径与四通道输出层。"""
        super().__init__()
        self.time_mlp = nn.Sequential(nn.Linear(32, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim))
        self.input_conv = nn.Conv2d(4, channels, 3, padding=1)
        self.down_block = MyResidualBlock(channels, channels * 2, time_dim)
        self.downsample = nn.Conv2d(channels * 2, channels * 2, 4, stride=2, padding=1)
        self.mid = MyResidualBlock(channels * 2, channels * 2, time_dim)
        self.upsample = nn.ConvTranspose2d(channels * 2, channels * 2, 4, stride=2, padding=1)
        self.up_block = MyResidualBlock(channels * 4, channels, time_dim)
        self.out = nn.Conv2d(channels, 4, 3, padding=1)

    def forward(self, x: torch.Tensor, timesteps: torch.Tensor):
        """根据时间步预测输入四通道潜变量中的噪声，输出张量与输入形状一致。"""
        time = self.time_mlp(my_timestep_embedding(timesteps, 32))
        skip = self.down_block(self.input_conv(x), time)
        hidden = self.mid(self.downsample(skip), time)
        hidden = self.upsample(hidden)
        hidden = torch.cat([hidden, skip], dim=1)
        return self.out(self.up_block(hidden, time))

unet = MyTinyUNet()
# 固定形状：latents.shape = [2, 4, 32, 32]。
latents = torch.randn(2, 4, 32, 32)
predicted_noise = unet(latents, torch.tensor([10, 500]))
print(predicted_noise.shape)


#### 3.3.2．`UNet2DConditionModel` 接口对照

本节使用完整的条件 U-Net 类型，并缩小通道数以检查接口。`encoder_hidden_states` 的最后一维模拟文本编码器宽度；生产 checkpoint 会从配置恢复全部真实尺寸和注意力层。


In [ ]:
# 配置 UNet2DConditionModel，接入标准条件扩散网络接口。

from diffusers import UNet2DConditionModel

# 32/64 通道与 8 维 Attention Head 是小型结构夹具；增大会提高容量、显存与延迟。
lib_unet = UNet2DConditionModel(
    sample_size=32,
    in_channels=4,
    out_channels=4,
    layers_per_block=1,
    block_out_channels=(32, 64),
    down_block_types=("CrossAttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "CrossAttnUpBlock2D"),
    cross_attention_dim=64,
    attention_head_dim=8,
    norm_num_groups=8,
)
# 准备持久化路径或状态对象，作为保存和重载的明确边界。
# 固定形状：lib_text_states.shape = [2, 16, 64]。
lib_text_states = torch.randn(2, 16, 64)
lib_noise_prediction = lib_unet(
    sample=latents,
    timestep=torch.tensor([10, 500]),
    encoder_hidden_states=lib_text_states,
).sample
print(lib_noise_prediction.shape)


### 3.4．训练目标：预测噪声

经典 SD v1 使用 $\epsilon$ prediction：随机采样图像、噪声和时间步，让 U-Net 预测刚加入的噪声，最小化 MSE。现代模型也可能使用 `v_prediction` 或 flow matching，不能混用目标。

#### 3.4.1．单步训练的从零实现


In [ ]:
# 随机采样时间步和噪声，使用预测噪声误差完成一次原理训练更新。

# lr=1e-4、Batch=2、weight_decay=0.01 和 grad clip=1.0 仅验证小模块更新路径。
# betas=(0.9,0.999) 与 eps=1e-8 显式固定优化器契约；真实训练从目标 Checkpoint 配方出发，频繁裁剪时先排查学习率与数值稳定性。
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=1e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
)
# 固定形状：clean_latents.shape = [2, 4, 32, 32]。
clean_latents = torch.randn(2, 4, 32, 32)
target_noise = torch.randn_like(clean_latents)
# 在 0…999 上均匀采样并使用未加权 MSE，构成最小 DDPM 目标；更换采样分布或加权方式须整体重验。
train_timesteps = torch.randint(0, num_steps, (2,))
noisy_latents = my_add_noise(clean_latents, target_noise, train_timesteps, alpha_bars)

# 清空上一轮梯度，避免 PyTorch 默认的梯度累积。
optimizer.zero_grad(set_to_none=True)
noise_prediction = unet(noisy_latents, train_timesteps)
loss = F.mse_loss(noise_prediction, target_noise)
# 反向传播计算梯度，供随后的参数更新使用。
loss.backward()
torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
optimizer.step()
print("loss:", float(loss.detach()))


#### 3.4.2．标准库训练接口对照

标准做法让 scheduler 负责加噪，让 `UNet2DConditionModel` 返回 `.sample`。训练 Stable Diffusion 还需冻结或微调 VAE/文本编码器，并使用 Accelerate 管理混合精度与分布式训练。


In [ ]:
# 沿用 Scheduler 与条件 U-Net 接口计算标准库训练损失。

# 整理当前阶段使用的数据子集，避免不同用途的数据混用。
lib_train_timesteps = torch.randint(0, lib_scheduler.config.num_train_timesteps, (2,))
lib_target_noise = torch.randn_like(latents)
lib_noisy_latents = lib_scheduler.add_noise(latents, lib_target_noise, lib_train_timesteps)
lib_prediction = lib_unet(
    lib_noisy_latents,
    lib_train_timesteps,
    encoder_hidden_states=lib_text_states,
).sample
lib_loss = F.mse_loss(lib_prediction, lib_target_noise)
print("library training loss:", float(lib_loss.detach()))


### 3.5．文本条件与 Cross-Attention

图像 latent 产生 query，文本 token 表示产生 key/value：

$$CrossAttention(Q_{image}, K_{text}, V_{text})$$

#### 3.5.1．Cross-Attention 的从零实现


In [ ]:
# 以图像特征作为 Query、文本特征作为 Key/Value，从零实现 Cross-Attention。

class MyCrossAttention(nn.Module):
    """以图像 Token 为 Query、文本 Token 为 Key/Value 实现多头交叉注意力。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, query_dim: int, context_dim: int, num_heads: int):
        """创建 Q、K、V 与输出投影，并记录头数和每头维度。"""
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = query_dim // num_heads
        self.q = nn.Linear(query_dim, query_dim, bias=False)
        self.k = nn.Linear(context_dim, query_dim, bias=False)
        self.v = nn.Linear(context_dim, query_dim, bias=False)
        self.out = nn.Linear(query_dim, query_dim)

    def my_heads(self, x: torch.Tensor):
        """把 [B,N,D] 张量重排为 [B,H,N,Dh] 多头布局。"""
        b, n, hidden_size = x.shape
        return x.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, image_tokens: torch.Tensor, text_tokens: torch.Tensor):
        """返回与图像 Token 对齐的条件特征及 [B,H,N_image,N_text] 注意力权重。"""
        # Query 来自图像，Key/Value 来自文本，因此输出仍与图像 token 对齐。
        q = self.my_heads(self.q(image_tokens))
        k = self.my_heads(self.k(text_tokens))
        v = self.my_heads(self.v(text_tokens))
        attention = (q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)).softmax(-1)
        context = attention @ v
        b, num_heads, n, head_dim = context.shape
        return self.out(context.transpose(1, 2).reshape(b, n, -1)), attention

# 77×768 是 Stable Diffusion v1 的 CLIP 文本接口，8 头须整除 64 维图像特征；其他模型应读取其 Config。
cross_attention = MyCrossAttention(64, 768, 8)
# 固定形状：image_tokens.shape = [2, 64, 64]。
image_tokens = torch.randn(2, 64, 64)
# 固定形状：text_tokens.shape = [2, 77, 768]。
text_tokens = torch.randn(2, 77, 768)
conditioned, attention_map = cross_attention(image_tokens, text_tokens)
print(conditioned.shape, attention_map.shape)


#### 3.5.2．CLIP Tokenizer 与 Text Encoder 接口对照

SD v1.x 使用 CLIP 文本编码器并固定最大长度 77。Pipeline 会自动完成 tokenization；训练或研究底层组件时可显式调用。


In [ ]:
# 用 CLIP Tokenizer 和 Text Encoder 把提示词转换为条件隐藏状态。

from transformers import CLIPTextModel, CLIPTokenizer

# 实例化当前阶段的模型结构，并准备进入训练或推理模式。
lib_text_model_id = "openai/clip-vit-large-patch14"
lib_tokenizer = CLIPTokenizer.from_pretrained(
    lib_text_model_id
)
lib_text_encoder = CLIPTextModel.from_pretrained(
    lib_text_model_id
).to(DEVICE)
lib_text_inputs = lib_tokenizer(
    ["a red panda reading a book"],
    padding="max_length",
    max_length=lib_tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt",
).input_ids.to(DEVICE)
lib_text_encoder.eval()
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.inference_mode():
    lib_prompt_embeddings = lib_text_encoder(lib_text_inputs).last_hidden_state
print(lib_prompt_embeddings.shape)


### 3.6．Classifier-Free Guidance（CFG）

同一 latent 分别做无条件与有条件预测，然后外推：

$$\epsilon_{cfg}=\epsilon_{uncond}+s(\epsilon_{cond}-\epsilon_{uncond})$$

$s$ 越大越贴合提示词，但过大会造成过饱和、伪影和多样性下降。

#### 3.6.1．CFG 公式的从零实现


In [ ]:
# 在线性外推中组合有条件与无条件噪声，得到 CFG 预测。

def my_classifier_free_guidance(unconditional, conditional, scale: float):
    """按给定 CFG Scale 从无条件预测沿条件差值方向线性外推。"""
    # 从无条件预测沿条件差值方向外推，scale 越大条件约束越强。
    return unconditional + scale * (conditional - unconditional)

# 固定形状：unconditional.shape = [1, 4, 32, 32]。
unconditional = torch.randn(1, 4, 32, 32)
conditional = torch.randn_like(unconditional)
# CFG=7.5 用于呈现提示遵循与画质权衡；增大可能过饱和，必须按目标 Checkpoint 的质量—延迟曲线校准。
guided = my_classifier_free_guidance(unconditional, conditional, scale=7.5)
print(guided.std())


#### 3.6.2．Pipeline 参数映射

Diffusers 在 `guidance_scale > 1` 时自动拼接无条件/有条件 batch，并在内部执行同一公式。


In [ ]:
# 把 guidance_scale 等生成参数集中为 Pipeline 调用配置。

# 30 步与 CFG=7.5 是 SD v1.5 接口基线；蒸馏、Flow 或其他 Checkpoint 不得直接复用。
lib_generation_options = {
    "prompt": "a red panda reading a book, cinematic lighting",
    "negative_prompt": "blurry, low quality, distorted",
    "guidance_scale": 7.5,
    "num_inference_steps": 30,
}
lib_generation_options


### 3.7．潜空间与 VAE

像素空间扩散成本很高。VAE 将 $512\times512\times3$ 图像编码为约 $64\times64\times4$ latent；扩散发生在 latent 中，最后再解码。训练时从后验分布重参数化采样：$z=\mu+\sigma\epsilon$。

#### 3.7.1．重参数化的从零实现


In [ ]:
# 用均值和对数方差重参数化采样潜变量，保持梯度可传播。

def my_reparameterize(mean: torch.Tensor, log_variance: torch.Tensor):
    """由均值和对数方差采样可重参数化潜变量；函数会消耗当前随机数状态。"""
    std = torch.exp(0.5 * log_variance)
    epsilon = torch.randn_like(std)
    return mean + std * epsilon

# 固定形状：mean.shape = [2, 4, 64, 64]。
mean = torch.zeros(2, 4, 64, 64)
log_variance = torch.zeros_like(mean)
z = my_reparameterize(mean, log_variance)
print(z.shape, float(z.std()))


#### 3.7.2．`AutoencoderKL` 接口对照

`scaling_factor` 是训练扩散模型时使用的 latent 缩放约定，必须从 VAE 配置读取，不能随意硬编码。


In [ ]:
# 切换到 AutoencoderKL，在图像空间与潜空间之间编码和解码。

from diffusers import AutoencoderKL

lib_vae = AutoencoderKL.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    subfolder="vae",
).to(DEVICE)
# 512×512 经该 VAE 的 8 倍空间压缩形成 64×64 Latent；缩放因子从 Config 读取，不硬编码 0.18215。
lib_pixels = torch.randn(1, 3, 512, 512, device=DEVICE).clamp(-1, 1)
lib_vae.eval()
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.inference_mode():
    lib_latent_dist = lib_vae.encode(lib_pixels).latent_dist
    lib_latents = lib_latent_dist.sample() * lib_vae.config.scaling_factor
    lib_reconstruction = lib_vae.decode(lib_latents / lib_vae.config.scaling_factor).sample
print("latent:", lib_latents.shape, "reconstruction:", lib_reconstruction.shape)


### 3.8．反向采样：从噪声到图像

采样器反复调用 U-Net 预测噪声，再根据 scheduler 的离散更新规则得到更干净的 latent。不同 scheduler（DDIM、Euler、DPM-Solver）在速度、稳定性与风格上有不同取舍。

#### 3.8.1．单步 DDPM 均值的从零实现

<!-- diagram:diffusion-sampling-loop -->
反向采样是 Scheduler 与 U-Net 反复协作的状态更新循环：

```mermaid
sequenceDiagram
    participant S as Scheduler
    participant U as Conditional U-Net
    participant L as Latent State
    participant V as VAE Decoder
    participant O as Image
    S->>L: 初始化 x_T
    loop t = T ... 1
        L->>U: x_t, t, text context
        U-->>S: predicted noise
        S->>L: step() 得到 x_(t-1)
    end
    L->>V: 最终 x_0 latent
    V-->>O: 解码为图像
```


In [ ]:
# 根据 DDPM 反向过程公式计算单步后验均值。

def my_ddpm_mean(xt, predicted_noise, timesteps, betas, alphas, alpha_bars):
    """根据当前噪声样本、预测噪声和扩散日程计算 DDPM 单步反向过程均值。"""
    beta_t = my_extract(betas, timesteps, xt.shape)
    alpha_t = my_extract(alphas, timesteps, xt.shape)
    alpha_bar_t = my_extract(alpha_bars, timesteps, xt.shape)
    return (xt - beta_t / torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_t)

reverse_mean = my_ddpm_mean(
    noisy_latents, noise_prediction.detach(), train_timesteps,
    betas, alphas, alpha_bars,
)
print(reverse_mean.shape)


#### 3.8.2．Scheduler Step 接口对照


## 4．证据验证

原理实现与库接口应提供以下证据：

1. 同一个 `x0` 和 `noise` 随时间步增大逐渐由噪声主导；`xt` 的形状始终与 `x0` 相同。
2. `my_add_noise` 与 `DDPMScheduler.add_noise` 在相同 Beta、时间步和噪声下的最大绝对误差接近当前精度舍入范围。
3. 原理实现与库时间嵌入的 Batch 维和目标维正确；U-Net 输出与输入 Latent 同形，训练损失为有限标量且梯度可回传。
4. CFG 满足 `scale=0` 返回无条件预测、`scale=1` 返回条件预测；增大 Scale 只改变线性外推，不保证质量单调提高。
5. VAE 编解码严格使用模型配置中的 `scaling_factor`；Scheduler 的 `prediction_type` 与训练目标一致。
6. 视频迁移时把图像 Latent `[B,C,H,W]` 扩为时空 Latent `[B,C,T,H,W]`，在容量报告中分别记录空间/时间压缩率、帧率、帧数和时空 Token。

这些证据证明数据流和接口成立，不证明生成内容已经达到业务质量；内容质量、安全和许可证在第 6 节单独验收。


In [ ]:
# 调用 Scheduler.step 更新潜变量，逐步从噪声走向样本。

# 30 个推理步控制质量与延迟；增加步数通常收益递减，Scheduler 变化后须重新标定。
lib_scheduler.set_timesteps(30)
lib_timestep = lib_scheduler.timesteps[0]
# 固定形状：lib_current.shape = [1, 4, 32, 32]。
lib_current = torch.randn(1, 4, 32, 32)
lib_noise = torch.randn_like(lib_current)
lib_previous = lib_scheduler.step(lib_noise, lib_timestep, lib_current).prev_sample
print(lib_previous.shape)


## 5．迁移到生产库

本节使用Stable Diffusion v1.5 的当前仓库文件 作为稳定接口基线，以观察组件映射；生产选型仍以第 2 节的模型图谱和业务验收为依据。真实权重执行由 `RUN_REAL_MODEL` 显式控制，默认值为 `False`，启用条件包括资源容量与模型许可均已核验。

| 原理实现/理论对象 | Diffusers 对象 | 图像/视频生产迁移 |
|---|---|---|
| 加噪、时间步与反向更新 | `Scheduler` | 固定训练目标、Scheduler 配置和求解器，不跨检查点随意替换 |
| 条件 U-Net | `UNet2DConditionModel` | 现代图像模型常改为 `Transformer2DModel`/专用 DiT；视频改为时空 Transformer |
| 图像 VAE | `AutoencoderKL` | 视频使用带时间压缩的 VAE；必须固定缩放、色域、分辨率与帧率协议 |
| 文本 Cross-Attention | Tokenizer + Text Encoder | 新模型可能使用多文本编码器、LLM 编码器或联合注意力 |
| 完整采样循环 | 任务专用 `Pipeline` | 按 T2I、Edit、T2V、I2V、Audio-Video 选择精确 Pipeline，不得仅替换 `MODEL_ID` |


In [ ]:
# 组合文本编码器、U-Net、VAE 与 Scheduler 形成完整 Stable Diffusion Pipeline。

from diffusers import EulerDiscreteScheduler, StableDiffusionPipeline

# 实例化当前阶段的模型结构，并准备进入训练或推理模式。
RUN_REAL_MODEL = False
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

if RUN_REAL_MODEL:
    lib_dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
    lib_pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=lib_dtype,
        use_safetensors=True,
    )
    lib_pipe.scheduler = EulerDiscreteScheduler.from_config(lib_pipe.scheduler.config)
    lib_pipe = lib_pipe.to(DEVICE)
    if DEVICE.type == "cuda":
        lib_pipe.enable_attention_slicing()
    lib_generator = torch.Generator(device=DEVICE.type).manual_seed(SEED)
    # 512×512、Euler、CFG=7.5 与 30 步共同构成生成协议；任一项变化均需固定 Seed 重验质量和显存。
    lib_image = lib_pipe(
        **lib_generation_options,
        generator=lib_generator,
        height=512,
        width=512,
    ).images[0]
    display(lib_image)
else:
    print("真实模型单元已就绪；将 RUN_REAL_MODEL=True 后执行。")


## 6．生产边界

1. **制品锁定**：Manifest 同时记录基础权重及哈希、VAE、文本编码器、Tokenizer、Scheduler、LoRA/Control 模块、精度、Pipeline 代码、许可证和 AUP；任何组件更新都重新回归。
2. **训练/微调**：全量训练成本极高；业务微调优先采用官方 LoRA/DreamBooth/视频微调脚本与 Accelerate。数据必须具备来源、许可、隐私、肖像/声音权利和内容安全审查，并保留 Caption/过滤规则。
3. **任务级评测**：图像覆盖提示遵循、文字/布局、编辑保持、画质、多样性、偏见、记忆化；视频增加运动、身份、镜头、时序、声画同步和物理一致性。FID/CLIP 等单指标不能代表可用性。
4. **服务边界**：网关限制提示长度、参考媒体、分辨率、帧率、帧数和时长；服务端实施显存预算、队列背压、超时、并发、租户隔离、可观测性、降级与幂等重试。
5. **安全与溯源**：输入输出双向审核；检测隐写/Prompt Injection；治理 NSFW、未成年人、肖像/声音冒充、商标和版权；生成内容添加水印/C2PA 或等价来源信息，并保留申诉/删除链路。
6. **开放性与商用**：分别核验代码和权重许可证、地域、收入/用户规模、衍生训练、再分发、归因与用途限制。API 可用、输出可商用和权重可商用是三个不同命题。

**常见风险**：把像素值范围 `[0,1]` 与 `[-1,1]` 混用、遗漏 VAE Scaling Factor、训练 Prediction Type 与 Scheduler 不一致、把图像 Attention 复杂度套到视频、FP16 在 CPU 上运行、CFG 过高、未经许可分发权重/数据或用受限输出训练其他基础模型。

### 6.1．官方资料

- 图像：[Qwen-Image](https://github.com/QwenLM/Qwen-Image)、[DeepSeek Janus](https://github.com/deepseek-ai/Janus)、[FLUX](https://github.com/black-forest-labs/flux)、[Stable Diffusion 3.5](https://github.com/Stability-AI/sd3.5)
- 视频：[Wan2.2](https://github.com/Wan-Video/Wan2.2)、[HunyuanVideo-1.5](https://github.com/Tencent-Hunyuan/HunyuanVideo-1.5)、[CogVideoX](https://github.com/zai-org/CogVideo)、[Mochi](https://github.com/genmoai/mochi)、[LTX-2](https://github.com/Lightricks/LTX-2)
- 世界模型：[NVIDIA Cosmos](https://github.com/NVIDIA/cosmos)
- 生产库：[Hugging Face Diffusers](https://github.com/huggingface/diffusers)


### 6.2．运行一致性边界

1. 同一噪声下，时间步增大应使信噪比按噪声日程下降，`$\bar\alpha_t$` 与 Scheduler 配置一致。
2. CFG 在 Scale 为 0 与 1 时应分别回到无条件和有条件预测；更高 Scale 不构成质量单调提升的保证。
3. 像素空间与 Latent 空间的元素数比例应与 VAE 空间压缩因子一致。
4. 更换 Scheduler 时应从原配置构造，并在固定 Seed、步数和输入下比较质量、耗时及数值差异。
